# SARSA no Labirinto
O agente parte do canto superior esquerdo e aprende a chegar ao canto inferior direito.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

random.seed(42)
np.random.seed(42)

## 1. Gerar o Labirinto

In [ ]:
SIZE = 20  # labirinto 20x20 (fácil de visualizar)

def gerar_labirinto(size, seed=42):
    random.seed(seed)
    maze = [[1]*size for _ in range(size)]  # 1 = parede
    
    def visitar(r, c):
        maze[r][c] = 0  # 0 = livre
        dirs = [(0,2),(0,-2),(2,0),(-2,0)]
        random.shuffle(dirs)
        for dr, dc in dirs:
            nr, nc = r+dr, c+dc
            if 0 <= nr < size and 0 <= nc < size and maze[nr][nc] == 1:
                maze[r+dr//2][c+dc//2] = 0  # abre parede entre
                visitar(nr, nc)
    
    visitar(1, 1)
    m = np.array(maze)
    m[0, :] = 1   # borda superior
    m[-1, :] = 1  # borda inferior
    m[:, 0] = 1   # borda esquerda
    m[:, -1] = 1  # borda direita
    return m

maze = gerar_labirinto(SIZE)
START = (1, 1)
GOAL  = (SIZE-2, SIZE-2)  # índice par = sempre livre

print(f"Labirinto {SIZE}x{SIZE} gerado. Início: {START}, Meta: {GOAL}")

## 2. Treinar o Agente com SARSA

In [ ]:
# Hiperparâmetros
alpha   = 0.2    # taxa de aprendizado
gamma   = 0.99   # desconto
epsilon = 1.0    # exploração inicial
episodios = 500

# Ações: cima, baixo, esquerda, direita
acoes = [(-1,0),(1,0),(0,-1),(0,1)]

# Q-table: (linha, coluna, ação)
Q = np.zeros((SIZE, SIZE, 4))

def escolher_acao(r, c, eps):
    if np.random.rand() < eps:
        return np.random.randint(4)
    return int(np.argmax(Q[r, c]))

def passo(r, c, a):
    dr, dc = acoes[a]
    nr, nc = r+dr, c+dc
    if 0 <= nr < SIZE and 0 <= nc < SIZE and maze[nr, nc] == 0:
        r, c = nr, nc
    recompensa = 100 if (r, c) == GOAL else -1
    return r, c, recompensa

recompensas = []

for ep in range(episodios):
    r, c = START
    a = escolher_acao(r, c, epsilon)
    total = 0

    for _ in range(2000):
        nr, nc, recomp = passo(r, c, a)
        na = escolher_acao(nr, nc, epsilon)

        # Equação SARSA
        Q[r,c,a] += alpha * (recomp + gamma * Q[nr,nc,na] - Q[r,c,a])

        r, c, a = nr, nc, na
        total += recomp
        if (r, c) == GOAL:
            break

    recompensas.append(total)
    epsilon = max(0.01, epsilon * 0.99)

print("Treinamento concluído!")

## 3. Extrair o Caminho Aprendido

In [ ]:
def caminho_greedy():
    r, c = START
    caminho = [(r, c)]
    visitados = {(r, c)}
    for _ in range(5000):
        a = int(np.argmax(Q[r, c]))
        nr, nc, _ = passo(r, c, a)
        if (nr, nc) in visitados:
            break
        r, c = nr, nc
        caminho.append((r, c))
        visitados.add((r, c))
        if (r, c) == GOAL:
            break
    return caminho

caminho = caminho_greedy()
chegou  = caminho[-1] == GOAL
print(f"Meta atingida: {'SIM' if chegou else 'NÃO'} | Passos: {len(caminho)}")

## 4. Visualização

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Labirinto + caminho ──────────────────────────────────────────
ax = axes[0]
ax.imshow(maze, cmap='binary', origin='upper')

# Caminho como linha colorida
if len(caminho) > 1:
    ys = [p[0] for p in caminho]
    xs = [p[1] for p in caminho]
    ax.plot(xs, ys, color='royalblue', linewidth=2, zorder=3)

# Início e meta
ax.plot(START[1], START[0], 'go', markersize=10, label='Início', zorder=5)
ax.plot(GOAL[1],  GOAL[0],  'r*', markersize=14, label='Meta',   zorder=5)

status = f"Chegou em {len(caminho)} passos" if chegou else "Não atingiu a meta"
ax.set_title(f'Labirinto {SIZE}×{SIZE} — {status}', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.axis('off')

# ── Curva de aprendizado ────────────────────────────────────────
ax2 = axes[1]
ax2.plot(recompensas, alpha=0.3, color='steelblue', linewidth=0.8)

janela = 20
media  = np.convolve(recompensas, np.ones(janela)/janela, mode='valid')
ax2.plot(range(janela-1, episodios), media, color='steelblue', linewidth=2,
         label=f'Média ({janela} ep.)')

ax2.set_title('Curva de Aprendizado — SARSA', fontsize=13)
ax2.set_xlabel('Episódio')
ax2.set_ylabel('Recompensa total')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Mapa de Q-Values
Quanto mais quente a célula, maior o valor que o agente aprendeu para ela.

In [ ]:
# Q-value máximo por célula
q_max = np.max(Q, axis=2)
q_max[maze == 1] = np.nan  # esconde paredes

fig, ax = plt.subplots(figsize=(7, 7))

im = ax.imshow(q_max, cmap='YlOrRd', origin='upper')
ax.imshow(maze, cmap='binary', origin='upper', alpha=0.4)  # sobrepos paredes

# Caminho
if len(caminho) > 1:
    ys = [p[0] for p in caminho]
    xs = [p[1] for p in caminho]
    ax.plot(xs, ys, color='royalblue', linewidth=2.5, zorder=3)

ax.plot(START[1], START[0], 'go', markersize=10, zorder=5)
ax.plot(GOAL[1],  GOAL[0],  'r*', markersize=14, zorder=5)

plt.colorbar(im, ax=ax, label='max Q(s, a)', shrink=0.8)
ax.set_title('Mapa de Q-Values — células mais quentes = mais valiosas', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

---
**Experimentos rápidos:**
- Aumente `episodios` para ver o agente melhorar
- Troque `SIZE = 20` por `SIZE = 30` para um labirinto maior
- Mude o `seed` em `gerar_labirinto()` para um labirinto diferente
- Reduza `episodios = 50` e veja como o caminho piora